# Baseline: vanilla Hunyuan3D-2 (no restyle)

Plain `tencent/Hunyuan3D-2` image-to-3D (shape + PBR paint), **no restyle** — the "theirs" baseline.

**Runtime → GPU (A100 recommended; paint needs ~16 GB VRAM).**

Flow: setup → load → run on each input image → download `baseline_glbs.zip`. Score locally with `eval/score_baseline.py` (same metrics).

Inputs: zip of your images (`eval/dataset/images/`), filenames matching `captions.csv`.

In [ ]:
!nvidia-smi -L

In [ ]:
# 1. Clone Hunyuan3D-2 + install (~15-20 min first time).
%cd /content
!git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
%cd /content/Hunyuan3D-2
import os
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.0;8.6;8.9;9.0'
os.environ['MAX_JOBS'] = '4'
!pip install diffusers==0.32.2 transformers==4.49.0 accelerate einops omegaconf \
    trimesh pymeshlab pygltflib xatlas opencv-python-headless 'numpy<2' \
    'tqdm>=4.66.3' rembg onnxruntime-gpu 'huggingface_hub[hf_transfer]'
!pip install -e . --no-deps
!cd hy3dgen/texgen/custom_rasterizer && python3 setup.py install
!cd hy3dgen/texgen/differentiable_renderer && python3 setup.py install

In [ ]:
# 2. Load shape + paint pipelines (downloads tencent/Hunyuan3D-2 weights).
%cd /content/Hunyuan3D-2
from hy3dgen.shapegen import (Hunyuan3DDiTFlowMatchingPipeline,
    FloaterRemover, DegenerateFaceRemover, FaceReducer)
from hy3dgen.texgen import Hunyuan3DPaintPipeline
from hy3dgen.rembg import BackgroundRemover

shape = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2', use_safetensors=True, device='cuda')
shape.enable_flashvdm(mc_algo='mc')
paint = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
rembg = BackgroundRemover()
floater, degen, reducer = FloaterRemover(), DegenerateFaceRemover(), FaceReducer()
print('pipelines ready')

In [ ]:
# 3. Upload images.zip (locally: cd eval/dataset && zip -r images.zip images)
from google.colab import files
import zipfile, os, glob
os.makedirs('/content/inputs', exist_ok=True)
for name in files.upload():
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('/content/inputs')
imgs = sorted(glob.glob('/content/inputs/**/*.png', recursive=True) +
              glob.glob('/content/inputs/**/*.jpg', recursive=True))
print(f'{len(imgs)} images')

In [ ]:
# 4. Vanilla Hunyuan3D-2 per input (NO restyle). Resumable.
from PIL import Image
import os, traceback, torch
OUT = '/content/baseline_glbs'; os.makedirs(OUT, exist_ok=True)
WITH_TEXTURE = True   # set False for geometry-only (much faster, less VRAM)

for i, p in enumerate(imgs, 1):
    stem = os.path.splitext(os.path.basename(p))[0]
    out_glb = f'{OUT}/{stem}.glb'
    if os.path.exists(out_glb): print(f'[{i}/{len(imgs)}] skip {stem}'); continue
    print(f'[{i}/{len(imgs)}] {stem} ...')
    try:
        img = Image.open(p).convert('RGB'); img = rembg(img)
        gen = torch.Generator(device='cuda').manual_seed(42)
        mesh = shape(image=img, num_inference_steps=50, octree_resolution=256,
                     guidance_scale=5.0, generator=gen, mc_algo='mc')[0]
        if WITH_TEXTURE:
            mesh = floater(mesh); mesh = degen(mesh)
            mesh = reducer(mesh, max_facenum=40000)
            mesh = paint(mesh, image=img)
        mesh.export(out_glb)
        print(f'    -> {out_glb}')
    except Exception:
        traceback.print_exc()

In [ ]:
# 5. Zip + download. Score locally:
#   .venv/bin/python eval/score_baseline.py eval/baseline_glbs_hunyuan eval/results_baseline_hunyuan.csv
import shutil
shutil.make_archive('/content/baseline_glbs', 'zip', '/content/baseline_glbs')
from google.colab import files
files.download('/content/baseline_glbs.zip')